Load the data from the CSV file into a pandas DataFrame using the following code:



In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



out = Path('../data/merged_school_dataset.parquet')
df = pd.read_parquet(out)

print(f"Merged dataset: {len(df):,} schools total")

Merged dataset: 3,439 schools total


In [2]:
# ─── 1. Creating Ofsted classification subset ────────────────────────────────────────────────
# Keep only schools with a usable graded judgement (1–4). NaN and 'Not judged' 
graded_mask = df["ofsted_grade"].astype(str).str.strip().isin(["1", "2", "3", "4"])
df_cls = df.loc[graded_mask].copy()
print(f"Classification subset: {len(df_cls):,} schools with graded Ofsted judgement")


# Keep the original for interpretation and reporting

grade_labels = ["Inadequate", "Requires improvement", "Good", "Outstanding"]

# To supprt CrossEntropyLoss label the Ofsted grades as 0–3 (1–4 in the original data)
# 3 = Outstanding, 2 = Good, 1 = RI, 0 = Inadequate
df_cls["ofsted_num"] = df_cls["ofsted_num"].astype(int)
y_cls = df_cls["ofsted_num"] - 1        # shift to 0-based for cross-entropy
print("Class distribution:\n", y_cls.value_counts().sort_index())


# ─── 2. Progress 8 REGRESSION subset ─────────────────────────────────────
#Regression uses Progress 8 for 2023/24 — the most recent year with published P8; 
# the 2024/25 file exists but has no P8 because the KS2 baseline was cancelled during COVID
df_reg = df.loc[df["p8_2024"].notna()].copy()
y_reg  = df_reg["p8_2024"]
print(f"Regression subset: {len(df_reg):,} schools with P8 2023/24")

# Attainment 8 regression subset (all three years available)
# Get 2023 and 2024 as train datasets and 2025 as test dataset.
train_2023 = df.loc[df["a8_2023"].notna()].copy()
train_2024 = df.loc[df["a8_2024"].notna()].copy()
test_2025  = df.loc[df["a8_2025"].notna()].copy()



#Audit the target variables to ensure they are clean and as expected
assert y_cls.notna().all(), "classification target still has NaN"
assert y_cls.isin([0, 1, 2, 3]).all(), "unexpected class labels"
# Expected value may drift with fresher extracts; because the ofsted data is updated monthly.
assert len(df_cls) == 1814, f"expected 1,814 graded schools, got {len(df_cls):,}"
print("\nTarget preparation complete.")


print(f"\nCls subset: {len(df_cls):,} schools | Reg subset: {len(df_reg):,} schools")
print("Class % of graded sample:",
      (y_cls.value_counts(normalize=True).sort_index() * 100).round(1).tolist())

Classification subset: 1,814 schools with graded Ofsted judgement
Class distribution:
 ofsted_num
0      55
1     251
2    1236
3     272
Name: count, dtype: int64
Regression subset: 3,141 schools with P8 2023/24

Target preparation complete.

Cls subset: 1,814 schools | Reg subset: 3,141 schools
Class % of graded sample: [3.0, 13.8, 68.1, 15.0]


Loading data ... Please wait while we fetch the necessary information.
